# 10 Explainability

This notebook prepares structured and text interpretation artifacts for the project report. It produces surrogate structured feature-importance tables, temporal importance summaries, note-phrase attention proxies, and clinician-facing explanation narratives.

## Explainability scope

- Structured feature importance tables
- Temporal feature importance by horizon
- Note phrase salience proxies from aggregated note windows
- Clinically interpretable narrative summaries
- Reusable CSV outputs for figures and final reporting

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [3]:
import pandas as pd

from src.evaluation.explainability import (
    build_attention_phrase_table,
    build_clinical_narrative_table,
    compute_surrogate_feature_importance,
    derive_temporal_feature_importance,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [4]:
config['explainability']

{'top_k_features': 20, 'top_k_phrases': 10, 'shap_sample_size': 200}

## Load baseline tabular artifacts and note-window tables

In [5]:
baseline_dir = paths['processed_data_dir'] / '06_baseline_models'
tabular_files = sorted(baseline_dir.glob('horizon_*_tabular.csv'))
assert tabular_files, 'No baseline tabular artifacts found. Run 06_baseline_models first.'

horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    structured_path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    if structured_path.exists():
        horizon_tables[dataset_name] = pd.read_csv(structured_path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'], low_memory=False)

note_window_path = paths['processed_data_dir'] / '05_text_processing' / 'horizon_6h_note_windows.csv'
note_windows_df = pd.read_csv(note_window_path) if note_window_path.exists() else pd.DataFrame()
len(tabular_files), {k: v.shape for k, v in horizon_tables.items()}, note_windows_df.shape

(3,
 {'horizon_6h': (1268294, 90),
  'horizon_12h': (1131670, 90),
  'horizon_24h': (860480, 90)},
 (114663, 12))

## Structured feature importance

This notebook uses surrogate correlation-based importance by default so it remains lightweight and dependency-safe. It can be replaced with SHAP once the final trained model objects are serialized.

In [6]:
tabular_df = pd.read_csv(tabular_files[0])
structured_importance_df = compute_surrogate_feature_importance(tabular_df).head(config['explainability']['top_k_features'])
structured_importance_df

,feature_name,importance
0,hours_to_prediction__min,0.074349
1,hours_to_prediction__last,0.074349
2,map__max__max,0.058850
3,map__max__last,0.053223
4,map__last__max,0.050704
5,map__mean__max,0.046916
6,map__mean__last,0.045030
7,map__last__last,0.044104
8,map__max__mean,0.041664
9,map__min__max,0.041619


## Temporal feature importance by horizon

In [7]:
temporal_importance_df = derive_temporal_feature_importance(
    horizon_tables=horizon_tables,
    top_k=config['explainability']['top_k_features'],
)
temporal_importance_df

,dataset_name,feature_name,importance
0,horizon_12h,map__max,0.036423
1,horizon_12h,map__mean,0.031089
2,horizon_12h,map__last,0.029940
3,horizon_12h,map__min,0.022869
4,horizon_12h,respiratory_rate__min,0.016874
5,horizon_12h,platelet__last,0.010693
6,horizon_12h,platelet__min,0.010689
7,horizon_12h,platelet__mean,0.010685
8,horizon_12h,platelet__max,0.010680
9,horizon_12h,respiratory_rate__mean,0.009092


## Note phrase salience proxy

When true attention weights from a trained text model are not yet serialized, this phrase-frequency proxy provides a lightweight first-pass interpretation table.

In [8]:
attention_phrase_df = build_attention_phrase_table(
    note_windows_df=note_windows_df,
    top_k_phrases=config['explainability']['top_k_phrases'],
)
attention_phrase_df

,phrase,pseudo_attention_score
0,with,0.269686
1,plan,0.087081
2,assessment,0.083747
3,this,0.083103
4,response,0.082811
5,______________________________________________...,0.080121
6,action,0.079860
7,right,0.079342
8,reason,0.077964
9,chest,0.076284


## Clinical interpretation narratives

In [9]:
clinical_narratives_df = build_clinical_narrative_table(
    structured_importance_df,
    top_k=min(config['explainability']['top_k_features'], 10),
)
clinical_narratives_df

,feature_name,clinical_interpretation
0,hours_to_prediction__min,This feature may reflect physiologic instabili...
1,hours_to_prediction__last,This feature may reflect physiologic instabili...
2,map__max__max,Blood pressure instability is clinically consi...
3,map__max__last,Blood pressure instability is clinically consi...
4,map__last__max,Blood pressure instability is clinically consi...
5,map__mean__max,Blood pressure instability is clinically consi...
6,map__mean__last,Blood pressure instability is clinically consi...
7,map__last__last,Blood pressure instability is clinically consi...
8,map__max__mean,Blood pressure instability is clinically consi...
9,map__min__max,Blood pressure instability is clinically consi...


## Save explainability artifacts

In [10]:
output_dir = paths['processed_data_dir'] / '10_explainability'
artifact_bundle = {
    'structured_feature_importance': structured_importance_df,
    'temporal_feature_importance': temporal_importance_df,
    'attention_phrase_proxy': attention_phrase_df,
    'clinical_narratives': clinical_narratives_df,
}
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'structured_feature_importance': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/10_explainability/structured_feature_importance.csv',
 'temporal_feature_importance': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/10_explainability/temporal_feature_importance.csv',
 'attention_phrase_proxy': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/10_explainability/attention_phrase_proxy.csv',
 'clinical_narratives': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/10_explainability/clinical_narratives.csv'}

In [11]:
manifest_path = paths['manifests_dir'] / '10_explainability_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='10_explainability',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'tabular_file_used': str(tabular_files[0]) if tabular_files else None,
        'note_window_available': bool(len(note_windows_df)),
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/10_explainability_manifest.json')

## Review checklist

For the final report, review:

- Which structured variables appear consistently important across horizons?
- Do the high-importance variables make clinical sense?
- Which note phrases recur in pre-sepsis windows?
- Are the explanation narratives specific enough for presentation and paper writing?
- Which artifacts should be converted into final figures?

## Project completion note

With this notebook in place, the staged Colab research pipeline is fully scaffolded from dataset setup through explainability. The remaining work is execution on the real MIMIC data, iteration on model quality, and final paper/report writing.